In [ ]:
import pandas as pd
from getpass import getpass
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

url = URL.create(
    "postgresql+psycopg2",
    username= input("PostgreSQL user: "),
    password=getpass("Postgres password: "),   # typed at run time, never saved in the notebook
    host="localhost",
    port=5432,                                  # change if docker ps shows a different host port
    database="retail",
)
engine = create_engine(url)

print(pd.read_sql("SELECT COUNT(*) AS rows FROM transactions", engine))   # expect 1033068

      rows
0  1033068


In [2]:
purchases = pd.read_sql("""
    SELECT customer_id, invoice,
           MIN(invoice_date) AS invoice_date,
           SUM(revenue)      AS invoice_revenue
    FROM transactions
    WHERE has_customer AND NOT is_cancellation
    GROUP BY customer_id, invoice
    HAVING SUM(revenue) > 0
""", engine, parse_dates=["invoice_date"])

print(purchases.shape)
print("customers:", purchases["customer_id"].nunique())      # expect 5852
print(purchases["invoice_date"].min(), purchases["invoice_date"].max())

(36595, 4)
customers: 5852
2009-12-01 07:45:00 2011-12-09 12:50:00


In [3]:
# one row per customer per purchase day (several invoices on one day count as one shopping occasion)
days = (purchases.assign(day=purchases["invoice_date"].dt.normalize())
        [["customer_id", "day"]].drop_duplicates()
        .sort_values(["customer_id", "day"]))

days["gap"] = days.groupby("customer_id")["day"].diff().dt.days
gaps = days["gap"].dropna()

print("purchase days:", len(days), "| gaps:", len(gaps))
print("customers with only one purchase day: %.1f%%"
      % (100 * days.groupby("customer_id").size().eq(1).mean()))
print(gaps.describe(percentiles=[.25, .5, .75, .9]).round(1))

for w in [30, 60, 90, 180]:
    print(f"share of gaps <= {w} days: {(gaps <= w).mean():.1%}")

purchase days: 32879 | gaps: 27027
customers with only one purchase day: 28.6%
count    27027.0
mean        59.3
std         78.3
min          1.0
25%         13.0
50%         32.0
75%         70.5
90%        148.0
max        714.0
Name: gap, dtype: float64
share of gaps <= 30 days: 49.0%
share of gaps <= 60 days: 70.8%
share of gaps <= 90 days: 80.6%
share of gaps <= 180 days: 92.7%


In [ ]:
cutoffs = pd.to_datetime(["2010-06-01", "2010-09-01", "2010-12-01",
                          "2011-03-01", "2011-06-01", "2011-09-01"])

rows = []
for c in cutoffs:
    known = pd.Series(purchases.loc[purchases["invoice_date"] < c, "customer_id"].unique())
    for w in (60, 90):
        end = c + pd.Timedelta(days=w)
        in_window = purchases["invoice_date"].between(c, end, inclusive="left")
        buyers = purchases.loc[in_window, "customer_id"].unique()
        label = known.isin(buyers)
        rows.append({"cutoff": c.date(), "window_days": w,
                     "customers": len(known), "buyers": int(label.sum()),
                     "base_rate": round(label.mean(), 3)})

print(pd.DataFrame(rows).to_string(index=False))

    cutoff  window_days  customers  buyers  base_rate
2010-06-01           60       2684    1116      0.416
2010-06-01           90       2684    1334      0.497
2010-09-01           60       3299    1528      0.463
2010-09-01           90       3299    1924      0.583
2010-12-01           60       4239    1130      0.267
2010-12-01           90       4239    1407      0.332
2011-03-01           60       4512    1240      0.275
2011-03-01           90       4512    1585      0.351
2011-06-01           60       4908    1351      0.275
2011-06-01           90       4908    1617      0.329
2011-09-01           60       5224    1705      0.326
2011-09-01           90       5224    2236      0.428
